# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/AliAlmasri7/FlyRank-ML/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Two paper findings + my methodology questions

I selected two findings from the FlyRank SEO research paper that are closely related to the signals used in my W05 model.

For each finding, I summarize the reported result and identify a methodology question about how strongly the evidence supports the interpretation. These questions are intended as validation checks rather than criticisms of the study.

### Finding #1 — The Anatomy of Growing Content

**Paper finding:**  
The paper reports that content trending upward tends to be longer, younger, and slightly better positioned than content trending downward. The growing cohort averaged about 3.2K words and 184 days of age, compared with 2.3K words and 230 days for the declining cohort. The sample contained 74,187 rising pages and 45,272 falling pages.

#### Methodology question

The main question I would ask is:

**How much of the observed difference between growing and declining content is associated with the way the growth/decline label is constructed, rather than being evidence that content age or word count itself drives future growth?**

The paper defines direction using the change in impressions between the most recent 30 days and the previous 30 days. This is useful for describing momentum, but the comparison is observational. Growing pages may differ from declining pages in other factors such as existing visibility, topic, client, competition, or previous performance.

Therefore, the finding supports an observed structural difference between the two groups, but it does not by itself establish that increasing word count or reducing content age would cause a page to move from declining to growing.

A stronger validation would use a time-aware design that measures features before the growth window and evaluates whether those features predict a later outcome while controlling for important grouping factors such as client.

### Finding #3 — Click Capture by Position Tier

**Paper finding:**  
The paper reports that weighted CTR declines as content moves farther away from the top search positions. The reported weighted CTR is 0.423% for the top 3, 0.339% for positions 4–10, 0.325% for positions 11–20, 0.163% for positions 21–50, and 0.050% for positions 50+.

#### Methodology question

The main question I would ask is:

**Does the observed relationship between position tier and CTR establish that improving a page's position will cause its CTR to increase by the same pattern?**

The paper calculates weighted CTR from aggregate clicks and impressions within each position tier, which is appropriate for a portfolio-level descriptive comparison. However, the comparison is observational: pages in different position tiers can differ in topic, search intent, brand strength, content quality, competition, and other factors.

Therefore, the result supports the measured association between visibility and click capture in this portfolio, but it should not be interpreted as a causal estimate of the CTR increase that would result from moving an individual page from one position tier to another.

A stronger validation would follow the same pages over time and measure whether changes in average position are followed by corresponding changes in CTR, ideally while accounting for client and other page-level differences.

# 2. My model under an honest split (before/after)

In W05, the Random Forest used a random 80/20 split of content rows. 
Although the features came from March and the target came from April, a random split can place content from the same client in both the training and validation sets.

For this validation audit, I use a client-grouped 80/20 split. 
All content belonging to a client is kept entirely in either the training or validation set.

This provides a stricter test of whether the model's observed signal transfers across clients.

In [2]:
import duckdb
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split, GroupShuffleSplit
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error
from scipy.stats import spearmanr

In [3]:
import os 
# Connect to DuckDB
con = duckdb.connect()

# Hugging Face token
HF_TOKEN = os.getenv("HF_TOKEN")

if HF_TOKEN is None:
    raise ValueError("HF_TOKEN was not found in the environment.")

# Create Hugging Face secret
con.execute(
    f"CREATE OR REPLACE SECRET hf_secret "
    f"(TYPE huggingface, TOKEN '{HF_TOKEN}')"
)

# Dataset location
rel = "hf://datasets/FlyRank/internship-warehouse"


In [23]:
march_query = f"""
SELECT
    client_hash_id,
    content_hash_id,
    gsc_impressions,
    gsc_clicks,
    gsc_avg_position
FROM read_parquet(
    '{rel}/fact_content_daily_performance/month=2026-03/*.parquet',
    hive_partitioning=true
)
"""

march_df = con.execute(march_query).df()

print("March rows:", len(march_df))
march_df.head()

March rows: 9841378


,client_hash_id,content_hash_id,gsc_impressions,gsc_clicks,gsc_avg_position
0,client_73cda7b4e4f265ea,content_b7e512995f79d5a6,20,0,3.350000
1,client_73cda7b4e4f265ea,content_05597932fe4da067,1,0,0.000000
2,client_73cda7b4e4f265ea,content_7a105f548d9c6916,125,1,4.928000
3,client_73cda7b4e4f265ea,content_905aa32a0230694e,7,0,4.000000
4,client_73cda7b4e4f265ea,content_a3ea9792f793ec72,11,0,2.272727


In [24]:
april_query = f"""
SELECT
    client_hash_id,
    content_hash_id,
    gsc_impressions AS future_impressions,
    gsc_clicks AS future_clicks
FROM read_parquet(
    '{rel}/fact_content_daily_performance/month=2026-04/*.parquet',
    hive_partitioning=true
)
"""

april_df = con.execute(april_query).df()

print("April rows:", len(april_df))
april_df.head()

April rows: 10424730


,client_hash_id,content_hash_id,future_impressions,future_clicks
0,client_62f4a7e64f5e0096,content_143987cfdeaba4c0,9,0
1,client_62f4a7e64f5e0096,content_13a8105125458098,1,0
2,client_62f4a7e64f5e0096,content_6a887d56ab6c8362,0,0
3,client_62f4a7e64f5e0096,content_e2bd76be7eed690d,1,0
4,client_62f4a7e64f5e0096,content_ddbfb1907979759a,0,0


In [25]:
march_df = (
    march_df
    .groupby(
        ["client_hash_id", "content_hash_id"],
        as_index=False
    )
    .agg({
        "gsc_impressions": "sum",
        "gsc_clicks": "sum",
        "gsc_avg_position": "mean"
    })
)

april_df = (
    april_df
    .groupby(
        ["client_hash_id", "content_hash_id"],
        as_index=False
    )
    .agg({
        "future_impressions": "sum",
        "future_clicks": "sum"
    })
)

print("March content-client rows:", len(march_df))
print("April content-client rows:", len(april_df))

March content-client rows: 331437
April content-client rows: 362172


In [26]:
model_df = march_df.merge(
    april_df,
    on=["client_hash_id", "content_hash_id"],
    how="left"
)

print("Joined rows:", len(model_df))

Joined rows: 331437


In [27]:
model_df["observed_ctr"] = np.where(
    model_df["gsc_impressions"] > 0,
    model_df["gsc_clicks"] / model_df["gsc_impressions"],
    np.nan
)

model_df["future_ctr"] = np.where(
    model_df["future_impressions"] > 0,
    model_df["future_clicks"] / model_df["future_impressions"],
    np.nan
)

In [28]:
model_df["future_ctr"].describe()

count    176441.000000
mean          0.003292
std           0.030496
min           0.000000
25%           0.000000
50%           0.000000
75%           0.001623
max           1.000000
Name: future_ctr, dtype: float64

In [29]:
model_df = model_df[
    model_df["future_impressions"] >= 10
].copy()

print("Rows after future impression threshold:", len(model_df))

Rows after future impression threshold: 146687


In [30]:
def assign_position_group(position):
    if pd.isna(position):
        return "Unknown"
    elif position <= 3:
        return "1-3"
    elif position <= 10:
        return "4-10"
    elif position <= 20:
        return "11-20"
    elif position <= 50:
        return "21-50"
    else:
        return "50+"

In [31]:
model_df["position_group"] = model_df["gsc_avg_position"].apply(
    assign_position_group
)

model_df["position_group"].value_counts()

position_group
4-10       61202
21-50      28890
11-20      27903
1-3        11315
Unknown     8945
50+         8432
Name: count, dtype: int64

In [32]:
required_cols = [
    "client_hash_id",
    "content_hash_id",
    "gsc_impressions",
    "gsc_clicks",
    "observed_ctr",
    "gsc_avg_position",
    "position_group",
    "future_ctr"
]

model_df = model_df.dropna(
    subset=required_cols
).copy()

print("Final modeling rows:", len(model_df))

Final modeling rows: 137742


In [33]:
feature_cols = [
    "gsc_impressions",
    "gsc_clicks",
    "observed_ctr",
    "gsc_avg_position",
    "position_group"
]

target_col = "future_ctr"

In [34]:
X = pd.get_dummies(
    model_df[feature_cols],
    columns=["position_group"],
    drop_first=False
)

y = model_df[target_col]

print("X shape:", X.shape)
print("y shape:", y.shape)

X shape: (137742, 9)
y shape: (137742,)


In [35]:
print("Missing values in X:")
print(X.isna().sum())

print("\nMissing values in y:")
print(y.isna().sum())

Missing values in X:
gsc_impressions         0
gsc_clicks              0
observed_ctr            0
gsc_avg_position        0
position_group_1-3      0
position_group_11-20    0
position_group_21-50    0
position_group_4-10     0
position_group_50+      0
dtype: int64

Missing values in y:
0


In [36]:
X_train_random, X_valid_random, y_train_random, y_valid_random = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42
)

print("Random training rows:", len(X_train_random))
print("Random validation rows:", len(X_valid_random))

Random training rows: 110193
Random validation rows: 27549


In [37]:
random_model = RandomForestRegressor(
    n_estimators=100,
    max_depth=12,
    min_samples_leaf=20,
    random_state=42,
    n_jobs=-1
)

random_model.fit(
    X_train_random,
    y_train_random
)

,n_estimators,100
,criterion,'squared_error'
,max_depth,12
,min_samples_split,2
,min_samples_leaf,20
,min_weight_fraction_leaf,0.0
,max_features,1.0
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,bootstrap,True
,oob_score,False


In [38]:
random_predictions = random_model.predict(
    X_valid_random
)

In [39]:
random_mae = mean_absolute_error(
    y_valid_random,
    random_predictions
)

random_mse = mean_squared_error(
    y_valid_random,
    random_predictions
)

random_rmse = np.sqrt(random_mse)

random_r2 = random_model.score(
    X_valid_random,
    y_valid_random
)

random_spearman = spearmanr(
    y_valid_random,
    random_predictions
).statistic

print(f"Random split MAE:       {random_mae:.6f}")
print(f"Random split RMSE:      {random_rmse:.6f}")
print(f"Random split R²:        {random_r2:.6f}")
print(f"Random split Spearman:  {random_spearman:.6f}")

Random split MAE:       0.002431
Random split RMSE:      0.006412
Random split R²:        0.113947
Random split Spearman:  0.354474


In [41]:
random_validation = model_df.loc[
    X_valid_random.index
].copy()

random_validation["ml_prediction"] = random_predictions

random_validation.head()

,client_hash_id,content_hash_id,gsc_impressions,gsc_clicks,gsc_avg_position,future_impressions,future_clicks,observed_ctr,future_ctr,position_group,ml_prediction
241309,client_73cda7b4e4f265ea,content_ecfe69245c77e3f6,406,0,44.120423,412.0,1.0,0.000000,0.002427,21-50,0.000530
65906,client_23a62021009f63c4,content_a9b68519d79b069b,110,1,2.184918,26.0,0.0,0.009091,0.000000,1-3,0.004371
3654,client_08a6a72ff48e62c0,content_1e339e9e31e6bd1f,4,0,37.000000,19.0,0.0,0.000000,0.000000,21-50,0.000659
44539,client_1a730cb2640a1abf,content_c84f29b184b14c42,152,0,7.315216,338.0,1.0,0.000000,0.002959,4-10,0.001604
177112,client_62f4a7e64f5e0096,content_15625ae27c50806f,98,0,53.271182,28.0,0.0,0.000000,0.000000,50+,0.000700


In [42]:
def top_k_mean_ctr(
    df,
    score_col="ml_prediction",
    target_col="future_ctr",
    k_values=(20, 50, 100, 500)
):
    rows = []

    ranked = df.sort_values(
        score_col,
        ascending=False
    )

    for k in k_values:
        top = ranked.head(
            min(k, len(ranked))
        )

        rows.append({
            "k": k,
            "mean_future_ctr": top[target_col].mean()
        })

    return pd.DataFrame(rows)

In [43]:
random_topk = top_k_mean_ctr(
    random_validation
)

random_topk

,k,mean_future_ctr
0,20,0.034417
1,50,0.030664
2,100,0.024015
3,500,0.014901


In [44]:
gss = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=42
)

train_idx, valid_idx = next(
    gss.split(
        X,
        y,
        groups=model_df["client_hash_id"]
    )
)

In [45]:
X_train_grouped = X.iloc[train_idx]
X_valid_grouped = X.iloc[valid_idx]

y_train_grouped = y.iloc[train_idx]
y_valid_grouped = y.iloc[valid_idx]

print("Grouped training rows:", len(X_train_grouped))
print("Grouped validation rows:", len(X_valid_grouped))

Grouped training rows: 107945
Grouped validation rows: 29797


In [46]:
train_clients = set(
    model_df.iloc[train_idx]["client_hash_id"]
)

valid_clients = set(
    model_df.iloc[valid_idx]["client_hash_id"]
)

client_overlap = train_clients.intersection(
    valid_clients
)

print("Training clients:", len(train_clients))
print("Validation clients:", len(valid_clients))
print("Client overlap:", len(client_overlap))

Training clients: 36
Validation clients: 9
Client overlap: 0


In [47]:
grouped_model = RandomForestRegressor(
    n_estimators=100,
    max_depth=12,
    min_samples_leaf=20,
    random_state=42,
    n_jobs=-1
)

grouped_model.fit(
    X_train_grouped,
    y_train_grouped
)

,n_estimators,100
,criterion,'squared_error'
,max_depth,12
,min_samples_split,2
,min_samples_leaf,20
,min_weight_fraction_leaf,0.0
,max_features,1.0
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,bootstrap,True
,oob_score,False


In [48]:
grouped_predictions = grouped_model.predict(
    X_valid_grouped
)

In [49]:
grouped_mae = mean_absolute_error(
    y_valid_grouped,
    grouped_predictions
)

grouped_mse = mean_squared_error(
    y_valid_grouped,
    grouped_predictions
)

grouped_rmse = np.sqrt(grouped_mse)

grouped_r2 = grouped_model.score(
    X_valid_grouped,
    y_valid_grouped
)

grouped_spearman = spearmanr(
    y_valid_grouped,
    grouped_predictions
).statistic

print(f"Grouped split MAE:       {grouped_mae:.6f}")
print(f"Grouped split RMSE:      {grouped_rmse:.6f}")
print(f"Grouped split R²:        {grouped_r2:.6f}")
print(f"Grouped split Spearman:  {grouped_spearman:.6f}")

Grouped split MAE:       0.002357
Grouped split RMSE:      0.006052
Grouped split R²:        0.123012
Grouped split Spearman:  0.378473


In [50]:
grouped_validation = model_df.iloc[
    valid_idx
].copy()

grouped_validation["ml_prediction"] = grouped_predictions

grouped_validation.head()

,client_hash_id,content_hash_id,gsc_impressions,gsc_clicks,gsc_avg_position,future_impressions,future_clicks,observed_ctr,future_ctr,position_group,ml_prediction
29084,client_0fa64a184f18a4a0,content_003959d7e5063269,464,1,8.768573,240.0,0.0,0.002155,0.0,4-10,0.002268
29085,client_0fa64a184f18a4a0,content_007c411f36b2ecb9,24,0,4.305556,48.0,0.0,0.000000,0.0,4-10,0.002894
29089,client_0fa64a184f18a4a0,content_00bbb13014fbe600,632,0,31.104595,1084.0,0.0,0.000000,0.0,21-50,0.000475
29090,client_0fa64a184f18a4a0,content_014620a9dadbad7b,1,0,0.000000,492.0,0.0,0.000000,0.0,1-3,0.002441
29091,client_0fa64a184f18a4a0,content_01adf4af9a86f26c,128,0,2.549947,19.0,0.0,0.000000,0.0,1-3,0.001403


In [51]:
grouped_topk = top_k_mean_ctr(
    grouped_validation
)

grouped_topk

,k,mean_future_ctr
0,20,0.046630
1,50,0.037173
2,100,0.025199
3,500,0.013091


In [52]:
random_topk_dict = dict(
    zip(
        random_topk["k"],
        random_topk["mean_future_ctr"]
    )
)

grouped_topk_dict = dict(
    zip(
        grouped_topk["k"],
        grouped_topk["mean_future_ctr"]
    )
)

In [54]:
comparison = pd.DataFrame([
    {
        "split": "W06 random 80/20",
        "validation_rows": len(X_valid_random),
        "validation_clients": model_df.loc[
            X_valid_random.index
        ]["client_hash_id"].nunique(),
        "spearman": random_spearman,
        "top_20_mean_future_ctr": random_topk_dict.get(20),
        "top_50_mean_future_ctr": random_topk_dict.get(50),
        "top_100_mean_future_ctr": random_topk_dict.get(100),
        "top_500_mean_future_ctr": random_topk_dict.get(500)
    },
    {
        "split": "W06 client-grouped 80/20",
        "validation_rows": len(X_valid_grouped),
        "validation_clients": model_df.iloc[
            valid_idx
        ]["client_hash_id"].nunique(),
        "spearman": grouped_spearman,
        "top_20_mean_future_ctr": grouped_topk_dict.get(20),
        "top_50_mean_future_ctr": grouped_topk_dict.get(50),
        "top_100_mean_future_ctr": grouped_topk_dict.get(100),
        "top_500_mean_future_ctr": grouped_topk_dict.get(500)
    }
])

comparison

,split,validation_rows,validation_clients,spearman,top_20_mean_future_ctr,top_50_mean_future_ctr,top_100_mean_future_ctr,top_500_mean_future_ctr
0,W06 random 80/20,27549,42,0.354474,0.034417,0.030664,0.024015,0.014901
1,W06 client-grouped 80/20,29797,9,0.378473,0.046630,0.037173,0.025199,0.013091


In [55]:
comparison_change = comparison.copy()

metrics = [
    "spearman",
    "top_20_mean_future_ctr",
    "top_50_mean_future_ctr",
    "top_100_mean_future_ctr",
    "top_500_mean_future_ctr"
]

for metric in metrics:
    comparison_change.loc[
        1,
        metric + "_change"
    ] = (
        comparison.loc[1, metric]
        - comparison.loc[0, metric]
    )

comparison_change

,split,validation_rows,validation_clients,spearman,top_20_mean_future_ctr,top_50_mean_future_ctr,top_100_mean_future_ctr,top_500_mean_future_ctr,spearman_change,top_20_mean_future_ctr_change,top_50_mean_future_ctr_change,top_100_mean_future_ctr_change,top_500_mean_future_ctr_change
0,W06 random 80/20,27549,42,0.354474,0.034417,0.030664,0.024015,0.014901,NaN,NaN,NaN,NaN,NaN
1,W06 client-grouped 80/20,29797,9,0.378473,0.046630,0.037173,0.025199,0.013091,0.023999,0.012213,0.006509,0.001184,-0.00181


## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*

## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.